<a href="https://colab.research.google.com/github/bsheese/387/blob/main/21_training/21_0_Training_Loop/21_0_1_Training_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training in Practice: Part 1
## The Training Loop

---

## What This Notebook Is About

In 19_0_1 you ran a training loop and were told to watch it rather than study it. Since then you've seen every idea inside it: the loss (19_1_4), slopes and `.backward()` (20_0_2), optimizer steps (20_0_3) and models built from `nn.Linear` layers (20_1_1). This notebook builds that loop again, one piece at a time, on a new dataset that the rest of unit 21 and 22_0 also use.

**What you will learn:**
1. The Fashion-MNIST dataset, and how images become rows of numbers for a network
2. How `TensorDataset` and `DataLoader` hand the training data to the model in shuffled batches
3. Why the model outputs raw scores and lets `nn.CrossEntropyLoss` apply softmax
4. The training loop and the evaluation step, line by line, including `model.train()` and `model.eval()`
5. How Adam and plain SGD compare on the same network
6. How to check the finished model on the test set

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

data = np.load(fetch('fashion10k.npz'))
CLASSES = list(data['classes'])

print({name: data[name].shape for name in ['x_train', 'x_val', 'x_test']})

def prepare(images, labels):
    # 28 x 28 pixels from 0-255  ->  784 numbers from 0 to 1, one row per image
    X = torch.tensor(images.reshape(len(images), -1) / 255, dtype=torch.float32)
    return X.to(device), torch.tensor(labels, dtype=torch.long).to(device)

---

## Section 1: Fashion-MNIST

**Fashion-MNIST** is a set of 70,000 small grayscale photos of clothing and shoes from the online retailer Zalando, released in 2017. Each image is 28 × 28 pixels and belongs to one of 10 classes. It was made as a harder replacement for MNIST, an older dataset of handwritten digits that had become too easy to tell much apart.

The course uses a fixed part of it, so everyone works with the same images: 10,000 training images (1,000 per class), 5,000 validation images (500 per class) and the official 10,000 test images. Each image is stored as a 28 × 28 array of whole numbers from 0 (black) to 255 (white).

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for k in range(10):
    examples = np.where(data['y_train'] == k)[0][:2]
    for row in range(2):
        axes[row, k].imshow(data['x_train'][examples[row]], cmap='gray_r')
        axes[row, k].axis('off')
    axes[0, k].set_title(CLASSES[k], fontsize=9)
plt.tight_layout()
plt.show()

Some classes are easy to tell apart (trousers, bags), and some are not: T-shirts, pullovers, coats and shirts are all upper-body clothing seen from the front, at very low resolution.

### Images as rows of numbers

The networks in this unit are built from `nn.Linear` layers, which take a row of numbers. So each image is **flattened**: its 28 rows of 28 pixels are laid end to end as one row of 784 numbers. The pixel values are also divided by 255, so they run from 0 to 1, for the same reason the penguin measurements were standardized in 20_0_3: inputs on a small, similar scale make training better behaved.

The `prepare` function in the setup cell does both, and moves the result to the device.

In [ ]:
X_train, y_train = prepare(data['x_train'], data['y_train'])
X_val, y_val = prepare(data['x_val'], data['y_val'])
X_test, y_test = prepare(data['x_test'], data['y_test'])
print('X_train:', X_train.shape, X_train.dtype, f'values {X_train.min():.0f} to {X_train.max():.0f}')
print('y_train:', y_train.shape, y_train.dtype, 'first labels:', y_train[:8].tolist())

Flattening throws something away: the network is no longer told which pixels are next to each other. Pixel 29 is directly below pixel 1, but to an `nn.Linear` layer it's just input number 29. The network can still learn which inputs matter together, but it has to learn it from the data. 22_0 uses layers that keep the image's shape and compares the result with the network built here.

---

## Section 2: Batches with `DataLoader`

The Pets loop in 19_0_1 worked through the training photos 32 at a time. Training on **batches** like this, rather than on all the data at once, has two advantages. A batch fits in memory however large the dataset is. And the weights get updated after every batch, so one pass through the data (an epoch) gives many small steps instead of one. Each step uses only part of the data, so its slopes are a little noisy, but on average they point the right way; that randomness is the "stochastic" in stochastic gradient descent.

Two PyTorch classes handle the batching. `TensorDataset` pairs each image with its label, and `DataLoader` deals them out in batches, in a new random order every epoch when `shuffle=True`.

In [ ]:
train_set = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

x_batch, y_batch = next(iter(train_loader))      # take one batch to look at
print('one batch:', x_batch.shape, y_batch.shape)
print('batches per epoch:', len(train_loader))

Each batch is 64 images and their 64 labels, and one epoch is 157 batches (the last one has only 16 images). `next(iter(...))` just takes the first batch so we can look at it; the training loop uses `for x, y in train_loader:` instead.

---

## Section 3: The Model and the Loss

The model is a hidden layer of 128 ReLU neurons and an output layer of 10 neurons, one per class:

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)
print(model)
print(f'{sum(p.numel() for p in model.parameters()):,} weights')

There's no softmax at the end. The model outputs 10 raw **scores** per image (often called **logits**), like the Pets model's 4 scores in 19_1_4. The loss function, `nn.CrossEntropyLoss`, applies softmax itself and then takes $-\log$ of the probability of the true class, averaged over the batch. Doing both steps together is more accurate with 32-bit numbers than computing probabilities first. When you want probabilities from this model, apply `.softmax(dim=1)` to its scores yourself.

A check that `nn.CrossEntropyLoss` is the cross-entropy loss from 19_1_4:

In [ ]:
loss_fn = nn.CrossEntropyLoss()

scores = model(x_batch)
by_pytorch = loss_fn(scores, y_batch)

probs = scores.softmax(dim=1)
p_true = probs[torch.arange(len(y_batch)), y_batch]      # probability of the true class, per image
by_hand = (-torch.log(p_true)).mean()
print(f'nn.CrossEntropyLoss: {by_pytorch.item():.4f}   by hand: {by_hand.item():.4f}')
print(f'-log(1/10) = {-np.log(0.1):.4f}')

They agree. Before training, the loss is close to $-\log(0.1) = 2.30$, which is what you'd get by giving each of the 10 classes a probability of about 0.1. That's a useful check on any new classifier: an untrained model should start near $-\log(1/\text{number of classes})$. A starting loss far from that usually means something is wrong with the data or the labels.

---

## Section 4: The Training Loop, Line by Line

The loop has two parts. **Training** goes through the training data once, a batch at a time, and updates the weights after each batch. **Evaluation** measures the loss and accuracy on a set of data without changing anything.

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()                          # put layers into training behavior
    for x, y in loader:                    # one batch at a time, in a new random order each epoch
        optimizer.zero_grad()              # clear the slopes from the last batch (20_0_2)
        loss = loss_fn(model(x), y)        # forward pass and loss for this batch
        loss.backward()                    # slopes for every weight (20_0_2)
        optimizer.step()                   # move every weight a little downhill (20_0_3)

def evaluate(model, X, y):
    model.eval()                           # put layers into evaluation behavior
    with torch.no_grad():                  # no slopes needed, so don't record anything (20_0_2)
        scores = model(X)
        loss = loss_fn(scores, y).item()
        accuracy = (scores.argmax(dim=1) == y).float().mean().item()
    return loss, accuracy

Two lines are new: `model.train()` and `model.eval()`. Some kinds of layers behave differently while training than while being evaluated. This model has none of them, so here the two calls change nothing, but they belong in every loop so that adding such a layer later doesn't silently break evaluation. 21_1_1 introduces the first one (dropout) and shows what goes wrong without them.

The `fit` function runs the two parts once per epoch and records the training and validation results after each epoch. The training results come from `evaluate` too, so both are measured the same way, at the end of the epoch.

In [ ]:
def fit(model, optimizer, epochs, loader=train_loader):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        train_one_epoch(model, loader, optimizer)
        for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val)]:
            loss, acc = evaluate(model, X, y)
            history[f'{name}_loss'].append(loss)
            history[f'{name}_acc'].append(acc)
    return history

def plot_history(histories):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for label, h in histories.items():
        epochs = range(1, len(h['val_loss']) + 1)
        line, = ax1.plot(epochs, h['val_loss'], label=f'{label}: validation')
        ax1.plot(epochs, h['train_loss'], '--', color=line.get_color(), label=f'{label}: training')
        ax2.plot(epochs, h['val_acc'], label=f'{label}: validation', color=line.get_color())
        ax2.plot(epochs, h['train_acc'], '--', color=line.get_color(), label=f'{label}: training')
    ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(fontsize=8)
    ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

Now train for 10 epochs with Adam, the optimizer from 19_0_1:

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

adam_history = fit(model, optimizer, epochs=10)
print(f"after 10 epochs: training accuracy {adam_history['train_acc'][-1]:.3f}, "
      f"validation accuracy {adam_history['val_acc'][-1]:.3f}")
plot_history({'Adam': adam_history})

The dashed lines are the training images and the solid lines the validation images. Both losses fall and both accuracies rise over the 10 epochs. The training accuracy is a few points above the validation accuracy, the small gap you'd expect from 19_1_1, and the validation loss is still falling, so the model hasn't started overfitting yet. 21_1_1 trains a larger network for much longer and watches the gap grow.

---

## Section 5: Adam and SGD

In 20_0_3 you wrote plain gradient descent, which PyTorch calls `torch.optim.SGD`: every weight moves by its slope times one learning rate. **Adam** also moves every weight against its slope, but it adjusts the size of each weight's step using that weight's recent slopes. Weights whose slopes have been small and consistent get relatively larger steps, and weights whose slopes jump around get smaller ones. In practice that makes Adam less sensitive to the choice of learning rate.

Here is the same network, from the same starting weights, trained with SGD at two learning rates:

In [ ]:
histories = {'Adam (lr 0.001)': adam_history}
for lr in [0.01, 0.1]:
    torch.manual_seed(0)
    model_sgd = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
    histories[f'SGD (lr {lr})'] = fit(model_sgd, torch.optim.SGD(model_sgd.parameters(), lr=lr), epochs=10)

for label, h in histories.items():
    print(f"{label:16} validation accuracy after 10 epochs: {h['val_acc'][-1]:.3f}")
plot_history(histories)

With a learning rate of 0.01, SGD is still a long way behind after 10 epochs; it's heading the right way, but slowly, like the small learning rate in 20_0_3. With 0.1 it does much better and gets close to Adam. So SGD can do well, but it needs its learning rate chosen with more care. That's why Adam with a learning rate of 0.001 is a common first choice, and it's what the rest of this unit uses.

---

## Section 6: The Test Set

All the choices in this notebook (the network, the optimizer, the number of epochs) were made by looking at validation results. The test images played no part, so the model's test accuracy is the honest estimate of how it does on new images (19_1_1). We use the Adam model, the one we'd pick from the validation results.

In [ ]:
test_loss, test_acc = evaluate(model, X_test, y_test)
print(f'test accuracy: {test_acc:.3f}  (on {len(y_test):,} images)')

with torch.no_grad():
    predicted = model(X_test).argmax(dim=1).cpu().numpy()
matrix = confusion_matrix(y_test.cpu().numpy(), predicted)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(matrix, display_labels=CLASSES).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title('Test images')
plt.show()

In [ ]:
right_per_class = np.diag(matrix) / matrix.sum(axis=1)
for k in np.argsort(right_per_class):
    print(f'{CLASSES[k]:12} {right_per_class[k]:.2f} of test images right')

The confusion matrix (19_1_2) shows where the mistakes are. The hardest class by far is **Shirt**, which the model mixes up with T-shirt/top, pullover and coat, the other upper-body classes. Trousers, bags, sandals and sneakers are almost always right. These are mostly mistakes a person squinting at a 28 × 28 image might also make, and they're the classes where seeing the shape of the garment (collar, sleeves, buttons) would help most. That's what 22_0's convolutional network is built to do.

---

## Summary

- **Fashion-MNIST** has 28 × 28 grayscale images of clothing in 10 classes. The course uses 10,000 training, 5,000 validation and 10,000 test images from it, the same for everyone.
- For an `nn.Linear` network each image is **flattened** to 784 numbers and scaled to the range 0 to 1. Flattening loses the information about which pixels are neighbors.
- `TensorDataset` pairs inputs with labels; `DataLoader` deals them out in shuffled **batches**, giving many weight updates per epoch.
- The model outputs raw **scores** (logits). `nn.CrossEntropyLoss` applies softmax and $-\log$ itself. An untrained classifier's loss should be near $-\log(1/\text{number of classes})$.
- The **training loop** for each batch: `optimizer.zero_grad()`, forward pass and loss, `loss.backward()`, `optimizer.step()`. **Evaluation** runs under `torch.no_grad()`. `model.train()` and `model.eval()` switch layers between their training and evaluation behavior.
- **Adam** adapts each weight's step size and works well with its usual learning rate (0.001); **SGD** can do nearly as well but needs a well-chosen learning rate.
- The network reaches about 85% on the test images. Most mistakes are among the upper-body classes, especially Shirt.

**Next:** 21_1_1 trains a much bigger network on fewer images until it overfits badly, then tries three fixes: dropout, weight decay and more data.